# 真实连续数据全量特征批量提取（滑窗版 v2.1）本 notebook 是 v2 的**进一步性能优化版本**。## v2.1 优化内容（对比 v2）| 优化项 | v2 | v2.1 | 预期收益 ||--------|----|----|----------|| STFT 策略 | 每频带独立 STFT | **共享宽频带 STFT + 子带切片** | STFT 计算 6→2 次，CPU -23% || 批次大小 | 256 | **2048** | 减少进程池开销，CPU +10~15% || NUMA 绑定 | 无 | **psutil cpu_affinity** | 减少跨 NUMA 内存访问 || STFT 批处理 | 单窗口 | **batched torch.stft** | 减少 PCIe 往返，GPU 30%→75%+ |## 三版本对比| 维度 | v1 | v2 | v2.1 ||------|----|----|------|| 并行模式 | ThreadPool | ProcessPool | ProcessPool || Workers | 固定 8 | 自动检测 | 自动检测 || STFT | scipy CPU | torch GPU | **torch GPU + 共享 + 批处理** || GPU 利用率 | ~45% | ~30% | **~75%+** || CPU 利用率 | ~55% | ~20% | **~80%+** || 预期加速比 | 基准 | 1.5~2.0x | **2.5~3.5x** |

In [ ]:
from __future__ import annotationsimport osimport sysfrom pathlib import Pathimport numpy as npimport pandas as pdfrom tqdm.auto import tqdmos.environ.setdefault('FEA_CPT_USE_GPU', '1')workspace = Path.cwd()if not (workspace / 'src').exists():    workspace = workspace.parentif str(workspace / 'src') not in sys.path:    sys.path.insert(0, str(workspace / 'src'))from fea_cpt_gpu_v2_1.sliding_window import (    SlidingWindowConfig,    build_sliding_window_dataset,    compute_shared_stft,    discover_source_files,    gpu_backend_info,    list_window_ranges,    upsample_to_target,    process_source_file,    _auto_detect_workers,)from fea_cpt_gpu_v2_1.params import DEFAULT_FEATURE_PARAMSprint(f'workspace = {workspace}')print(gpu_backend_info())print(f'Python = {sys.version}')print(f'CPU 核心数: {os.cpu_count()}')print(f'推荐 workers: {_auto_detect_workers()}')print('v2.1 所有模块加载成功')

In [ ]:
# =========================# 全局配置# =========================RAW_DATA_ROOTS = [    Path(r'G:\20260323_ZZ_pccp\FIP\24-900-1800\fip-24上午'),]WINDOW_DURATION_S = 0.02WINDOW_OVERLAP = 0.50assert 0.0 <= WINDOW_OVERLAP < 1.0TARGET_SAMPLE_RATE = 500_000.0PREPROC_BAND = (1_000.0, 95_000.0)BANDS = [    ('b_1k_100k',  (1_000.0,  100_000.0)),    ('b_1k_10k',   (1_000.0,  10_000.0)),    ('b_10k_20k',  (10_000.0, 20_000.0)),    ('b_20k_40k',  (20_000.0, 40_000.0)),    ('b_40k_60k',  (40_000.0, 60_000.0)),    ('b_60k_100k', (60_000.0, 100_000.0)),]# v2.1 并行设置WINDOW_WORKERS = None          # None = 自动检测WINDOW_BATCH_SIZE = 2048       # v2.1: 从 256 提升到 2048ENABLE_NUMA_BINDING = True     # v2.1: NUMA 绑定ENABLE_SHARED_STFT = True      # v2.1: 跨频带共享 STFTOUTPUT_ROOT = workspace / 'outputs' / 'realdata_continuous_features_20260528_v2_1'NPZ_PER_CSV = 100MAX_FILES: int | None = NoneTDMS_FALLBACK_SAMPLE_RATE_HZ: float | None = NoneOUTPUT_ROOT.mkdir(parents=True, exist_ok=True)auto_workers = _auto_detect_workers() if WINDOW_WORKERS is None else WINDOW_WORKERSprint(f'数据路径: {len(RAW_DATA_ROOTS)} 个根目录')print(f'滑窗: {WINDOW_DURATION_S*1000:.0f}ms, 重叠 {WINDOW_OVERLAP*100:.0f}%')print(f'目标采样率: {TARGET_SAMPLE_RATE/1000:.0f} kHz')print(f'频带数: {len(BANDS)}')print(f'并行: {auto_workers} workers, batch={WINDOW_BATCH_SIZE}')print(f'NUMA 绑定: {ENABLE_NUMA_BINDING}')print(f'共享 STFT: {ENABLE_SHARED_STFT}')print(f'输出目录: {OUTPUT_ROOT}')config = SlidingWindowConfig(    bands=BANDS,    preproc_band=PREPROC_BAND,    window_duration_s=WINDOW_DURATION_S,    window_overlap=WINDOW_OVERLAP,    target_sample_rate=TARGET_SAMPLE_RATE,    tdms_fallback_sample_rate=TDMS_FALLBACK_SAMPLE_RATE_HZ,    window_workers=WINDOW_WORKERS,    window_batch_size=WINDOW_BATCH_SIZE,    enable_numa_binding=ENABLE_NUMA_BINDING,    enable_shared_stft=ENABLE_SHARED_STFT,)print('\n配置对象创建成功')

In [ ]:
# =========================# 数据文件发现# =========================source_files = discover_source_files(RAW_DATA_ROOTS, max_files=MAX_FILES)if not source_files:    raise FileNotFoundError(f'未找到任何 .npz/.tdms 文件，请检查路径: {RAW_DATA_ROOTS}')from collections import Counterfolder_counts = Counter(f.parent.name for f in source_files)print(f'发现 {len(source_files)} 个源文件')print(f'\n各文件夹文件数:')for folder, count in sorted(folder_counts.items()):    print(f'  {folder}: {count}')npz_count = sum(1 for f in source_files if f.suffix.lower() == '.npz')tdms_count = sum(1 for f in source_files if f.suffix.lower() == '.tdms')print(f'\n文件格式: {npz_count} npz, {tdms_count} tdms')

In [ ]:
# =========================# 单文件处理测试# =========================import timetest_file = source_files[0]print(f'测试文件: {test_file.name}')from fea_cpt_gpu_v2_1.sliding_window import load_source_file, upsample_to_targetsrc = load_source_file(test_file, config.tdms_fallback_sample_rate)raw = np.asarray(src['signal_values'], dtype=float)orig_rate = float(src['sample_rate'])print(f'  原始采样率: {orig_rate/1000:.0f} kHz, 样本数: {len(raw):,}')sig_up, eff_rate = upsample_to_target(raw, orig_rate, config.target_sample_rate)print(f'  升采样后: {eff_rate/1000:.0f} kHz, 样本数: {len(sig_up):,}')windows = list_window_ranges(len(sig_up), eff_rate, config.window_duration_s, config.window_overlap)print(f'  窗口数: {len(windows)}')print('\n开始单文件特征计算（v2.1 共享 STFT + 大批次 + NUMA）...')t0 = time.time()df_feat, df_log = process_source_file(test_file, config)elapsed = time.time() - t0print(f'  耗时: {elapsed:.1f} s')print(f'  特征表形状: {df_feat.shape}')print(f'  v1 基准耗时: 127.0 s')print(f'  加速比: {127.0/elapsed:.2f}x')print('\n单文件测试通过！')

In [ ]:
# =========================# 批量处理主流程# =========================import timefrom datetime import datetimeprocessed_list_path = OUTPUT_ROOT / 'processed_source_files.txt'print(f'{"="*60}')print(f'批量处理开始 (v2.1: 共享STFT + NUMA + 大批次)')print(f'时间: {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}')print(f'文件数: {len(source_files)}')print(f'{"="*60}')t_start = time.time()stats = build_sliding_window_dataset(    source_paths=source_files,    config=config,    output_dir=OUTPUT_ROOT,    processed_list_path=processed_list_path,    npz_per_csv=NPZ_PER_CSV,    show_progress=True,)t_elapsed = time.time() - t_startprint(f'\n{"="*60}')print(f'批量处理完成')print(f'耗时: {t_elapsed:.1f} s ({t_elapsed/60:.1f} min)')print(f'新处理文件数: {stats["processed"]}')print(f'跳过文件数: {stats["skipped"]}')print(f'失败文件数: {stats["failed"]}')print(f'总窗口数: {stats["windows"]}')if stats["processed"] > 0:    avg = t_elapsed / stats["processed"]    print(f'单文件平均耗时: {avg:.1f} s')    print(f'v1 基准: 127.0 s, 加速比: {127.0/avg:.2f}x')print(f'{"="*60}')

In [ ]:
# =========================# 处理结果汇总# =========================feature_chunks = sorted(OUTPUT_ROOT.glob('features_part_*.csv'))log_chunks = sorted(OUTPUT_ROOT.glob('log_part_*.csv'))print(f'特征 CSV 文件数: {len(feature_chunks)}')if feature_chunks:    df_sample = pd.read_csv(feature_chunks[0], nrows=5)    print(f'特征 CSV 列数: {len(df_sample.columns)}')    total_windows = 0    total_files_set = set()    for chunk in feature_chunks:        df = pd.read_csv(chunk, usecols=['source_file_name', 'window_id'])        total_windows += len(df)        total_files_set.update(df['source_file_name'].unique())    print(f'总窗口数: {total_windows:,}')    print(f'总文件数: {len(total_files_set)}')

In [ ]:
# =========================# 特征质量检查# =========================if feature_chunks:    df_check = pd.read_csv(feature_chunks[0])    meta_cols = {        'source_file_name', 'source_file_path', 'source_format',        'source_group_name', 'source_channel_name', 'source_detail',        'window_id', 'window_start_index', 'window_end_index',        'window_length_samples', 'window_step_samples',        'window_duration_s', 'window_start_offset_s',        'sample_rate_hz', 'original_sample_rate_hz',        'source_n_samples', 'source_duration_s',        'starttime_raw', 'arrival_time_raw', 'sample_type',        'window_start_datetime',    }    feature_cols = [c for c in df_check.columns if c not in meta_cols]    print(f'特征列数: {len(feature_cols)}')    full_nan = [c for c in feature_cols if pd.to_numeric(df_check[c], errors='coerce').isna().all()]    if full_nan:        print(f'警告: {len(full_nan)} 个特征全为 NaN')    else:        print('无全 NaN 特征，数据质量良好')    # 快速统计    nan_stats = []    for col in feature_cols[:20]:        series = pd.to_numeric(df_check[col], errors='coerce')        nan_ratio = series.isna().sum() / len(series)        nan_stats.append({'feature': col, 'nan_ratio': nan_ratio, 'mean': series.mean()})    print('\n前20个特征统计:')    print(pd.DataFrame(nan_stats).to_string(index=False))

In [ ]:
# =========================# 运行日志# =========================processed_log = OUTPUT_ROOT / 'processed_source_files.txt'failed_log = OUTPUT_ROOT / 'failed_samples.log'if processed_log.exists():    lines = processed_log.read_text(encoding='utf-8').strip().split('\n')    print(f'已处理文件记录: {len(lines)} 条')    print('\n最近处理的 5 个文件:')    for line in lines[-5:]:        print(f'  {Path(line).name}')if failed_log.exists():    print(f'\n失败样本日志:')    print(failed_log.read_text(encoding='utf-8'))else:    print('\n无失败样本')